# Expert Choice Routing — a toy-scale build

A minimal implementation of **Expert Choice routing**, from Zhou, Lei, Liu,
Du, Huang, Zhao, Dai, Chen, Le, Laudon, *"Mixture-of-Experts with Expert
Choice Routing"* (2022) — a way to route tokens to experts that guarantees
perfectly even expert load *by construction*, instead of relying on an
auxiliary loss to encourage it.

Companion write-up: `README.md` in this folder. (See `../moe` for the more
common token-choice routing this notebook inverts.)

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. Flipping who does the choosing

The `../moe` folder in this repo implements the standard MoE routing
recipe: every **token** looks at all the experts and picks its favorite
top-k. This works, but nothing about it guarantees experts end up with
balanced workloads — some experts can end up popular and overloaded, others
neglected, which is why that notebook needs an auxiliary load-balancing
loss to actively discourage the router from collapsing onto a few experts.

**Expert Choice routing flips the direction of choice.** Instead of each
token picking its experts, each **expert** picks its favorite tokens:

```
for each expert e:
    score every token's affinity for expert e
    expert e picks its top-`capacity` favorite tokens
    process exactly those tokens, weighted by their affinity score
```

Since every expert always picks exactly `capacity` tokens — no more, no
less — **every expert is guaranteed to process the same number of tokens**,
by construction. There's no possibility of load imbalance for the auxiliary
loss to fix, because the mechanism itself can't produce imbalance in the
first place.

## 2. What "capacity" means here

`capacity` is simply how many tokens each expert is allowed to pick,
computed from a **capacity factor**:

```
capacity = capacity_factor * (total_tokens / n_experts)
```

A `capacity_factor` of `1.0` means "on average, each expert gets exactly
its fair share of tokens" (the same average load you'd get with token-choice
top-1 routing). A `capacity_factor` above `1.0` (this notebook uses `2.0`)
gives each expert some slack to pick a few more tokens than its exact fair
share, since some tokens are more universally useful to route on than
others and a hard 1x cap would waste some of that available compute
headroom.

## 3. What this means for individual tokens

There's a real trade-off buried in this design: because experts pick
tokens (not the other way around), it's now possible for a token to be
picked by **several** experts, or by **none at all** — nothing guarantees
every token gets processed. In a full model, this token-level unevenness is
usually fine, since the surrounding residual connection means an
un-routed token simply skips the MoE layer's contribution for that step,
rather than being dropped from the sequence outright.

> **Simplification used here:** the paper's routing uses a slightly more
> involved auxiliary procedure for very small `capacity_factor` settings
> (to keep training stable when the capacity is tight), and discusses batch
> composition effects when serving at inference time (since at inference,
> "top-c tokens out of this expert's column of scores" depends on which
> other tokens happen to be in the same batch — a wrinkle token-choice
> routing doesn't have, since each token's routing decision there doesn't
> depend on any other token in the batch). This notebook implements the
> core selection mechanism directly, with a generous `capacity_factor=2.0`
> that keeps training simple and stable at toy scale.

In [ ]:
class Expert(nn.Module):
    def __init__(self, d, hidden_mult=4):
        super().__init__()
        h = d * hidden_mult
        self.w1 = nn.Linear(d, h, bias=False)
        self.w2 = nn.Linear(d, h, bias=False)
        self.w3 = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class ExpertChoiceMoE(nn.Module):
    def __init__(self, d_model=64, n_experts=8, capacity_factor=2.0, hidden_mult=4):
        super().__init__()
        self.n_experts = n_experts
        self.capacity_factor = capacity_factor
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([Expert(d_model, hidden_mult) for _ in range(n_experts)])

    def forward(self, x):
        B, T, D = x.shape
        xf = x.reshape(-1, D)
        N = xf.shape[0]
        logits = self.router(xf)                        # N, E
        probs = logits.softmax(dim=0)                     # softmax OVER TOKENS, per expert (column-wise) --
                                                            # each expert's scores across all N tokens sum to 1
        capacity = max(1, int(self.capacity_factor * N / self.n_experts))   # exactly how many tokens each expert takes

        out = torch.zeros_like(xf)
        for e in range(self.n_experts):
            expert_scores = probs[:, e]                                    # N -- this expert's affinity for every token
            top_val, top_idx = expert_scores.topk(min(capacity, N))          # expert PICKS its favorite tokens
            weight = top_val / top_val.sum().clamp_min(1e-9)                 # renormalize over the tokens it picked
            out[top_idx] += weight.unsqueeze(-1) * self.experts[e](xf[top_idx])

        return out.reshape(B, T, D)     # no auxiliary loss needed -- balance is exact by construction

## 4. Checking the balance guarantee directly

Before wiring this into a language model, a quick, direct check that every
expert really does handle exactly `capacity` tokens, with no auxiliary loss
involved anywhere.

In [ ]:
m_check = ExpertChoiceMoE(d_model=64, n_experts=8, capacity_factor=2.0)
xf = torch.randn(40, 64)
logits = m_check.router(xf)
probs = logits.softmax(dim=0)
cap = max(1, int(m_check.capacity_factor * 40 / m_check.n_experts))
print(f"tokens = 40, experts = 8, capacity_factor = 2.0  ->  capacity per expert = {cap}")
print(f"total (expert, token) slots used = {cap * m_check.n_experts}  (a token can be picked more than once, or never)")
for e in range(m_check.n_experts):
    n_picked = min(cap, 40)
    print(f"  expert {e}: picks exactly {n_picked} tokens")

## 5. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32):
        super().__init__()
        self.h, self.dh = n_heads, d_head
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.scale = d_head ** -0.5

    def forward(self, x):
        B, T, D = x.shape
        H, Dh = self.h, self.dh
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, Dh).transpose(1, 2)
        attn = torch.einsum('bhtd,bhsd->bhts', q, k) * self.scale
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        attn = attn.masked_fill(mask, float('-inf')).softmax(-1)
        o = torch.einsum('bhts,bhsd->bhtd', attn, v).transpose(1, 2).reshape(B, T, H * Dh)
        return self.out_proj(o)

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.attns = nn.ModuleList([CausalSelfAttention(d_model) for _ in range(n_layers)])
        self.moes = nn.ModuleList([ExpertChoiceMoE(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for attn, moe, n1, n2 in zip(self.attns, self.moes, self.norms1, self.norms2):
            x = x + attn(n1(x))
            x = x + moe(n2(x))     # no auxiliary loss to add here, unlike ../moe
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through Expert Choice routing once it's wired into a real model. So the rest of this
notebook:

1. wraps Expert Choice routing into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Log which tokens go unprocessed.** Add a counter for how many of the
  `N` tokens in a batch get picked by zero experts at a given
  `capacity_factor`, and see how that count shrinks as you raise the
  capacity factor toward (and past) `2.0`.
- **Compare directly against `../moe`.** Same underlying idea (route
  tokens to a subset of experts), opposite direction of choice, and a
  completely different way of achieving load balance — one via an
  auxiliary loss, one by construction. Worth training both on the same toy
  task and comparing loss curves.
- **Try a very low `capacity_factor`** (e.g. `0.5`) and watch training
  degrade as more and more tokens get skipped by every expert — a direct,
  hands-on way to see why the capacity factor matters.

Reference: Zhou, Lei, Liu, Du, Huang, Zhao, Dai, Chen, Le, Laudon,
*"Mixture-of-Experts with Expert Choice Routing,"* 2022.